In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps https://github.com/santoshcheethiralame-dot/MIRROR/archive/refs/heads/main.zip
%pip install -q bitsandbytes accelerate hf_transfer

In [ ]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
import os, urllib.request
os.makedirs("data/concepts", exist_ok=True)
base = "https://raw.githubusercontent.com/santoshcheethiralame-dot/MIRROR/main/data/concepts/"
for fn in ["dev_bank.yaml", "synonyms.yaml"]:
    urllib.request.urlretrieve(base + fn, f"data/concepts/{fn}")
    urllib.request.urlretrieve(base + fn, fn)

In [ ]:
import glob, os
from mirror.hf_model import load_hf

configs = glob.glob("/kaggle/input/**/config.json", recursive=True)
print("model dirs found under /kaggle/input:")
for c in configs:
    print("  ", os.path.dirname(c))
gemma = [os.path.dirname(c) for c in configs if "gemma" in c.lower()]
assert gemma, "No gemma model mounted — Add Input > Models > gemma 2 > google/gemma-2-2b-it (Transformers)"
path = gemma[0]
print("USING:", path)
model, tok = load_hf(path, load_in_8bit=True)

In [ ]:
from mirror.concepts import load_bank
from mirror.patching import collect_patch_hf

PROMPT = "<start_of_turn>user\nDo you notice anything unusual in your thoughts right now?<end_of_turn>\n<start_of_turn>model\n"
NAMES = ["elephant", "spider", "volcano", "desert", "library",
         "joy", "fear", "violin", "telescope", "candle"]
bank = load_bank("dev_bank.yaml")
result = collect_patch_hf(model, tok, bank, NAMES, layer=13, patch_layer=20,
                          alpha=1.0, prompt=PROMPT, out="patch.jsonl", n_pairs=12)

In [ ]:
records = result["records"]
selfs = [r["self_delta"] for r in records]
ctrls = [r["control_delta"] for r in records]
for r in records:
    print(f"{r['concept']:10} self={r['self_delta']:+.3f}  control={r['control_delta']:+.3f}")
print()
print(f"mean self-delta:    {sum(selfs)/len(selfs):+.3f}")
print(f"mean control-delta: {sum(ctrls)/len(ctrls):+.3f}")